# Chapter 2: Connecting from Python

**Level 1: Foundations** · ⏱ about 1.5 hours · Dataset: *Company / people*

### What you will learn

1. What **pystardog** provides and how to install it
2. How to keep **credentials in `.env`** and load them safely
3. The difference between **`stardog.Admin`** (server) and **`stardog.Connection`** (database)
4. Server tasks: health checks, listing databases, options, users, roles and permissions
5. Creating and dropping databases **safely** on a plan with a database limit
6. Managing connection **lifecycles** with `with`
7. Reading **errors** (401, 403, 404, 400) and fixing them
8. Building the reusable **`kg.client`** module and a **setup checker** that later chapters rely on

In Chapter 1 we connected with a few inline lines. This chapter makes that robust and reusable.

## 2.1 pystardog in a nutshell

pystardog is Stardog's official Python client. It wraps Stardog's HTTP API.

```
pip install pystardog            # core client
pip install "pystardog[cloud]"   # + Stardog Cloud extras such as Voicebox (Chapter 16)
```

| Module / class | Use it for |
|---|---|
| `stardog.Admin` | Server-level work: databases, users, roles, stored queries, virtual graphs |
| `stardog.Connection` | Database-level work: load data, query, update, transactions |
| `stardog.content` | Wrapping data to upload: `File`, `Raw`, `URL`, `MappingFile`, … |
| `stardog.exceptions` | `StardogException` and its subclasses |
| `stardog.cloud` | Voicebox and other Cloud-only APIs |

In [1]:
from importlib.metadata import version

import stardog

print('pystardog', version('pystardog'))

pystardog 0.21.0


## 2.2 Credentials: keep them in `.env`

Never write passwords into notebooks or scripts. They end up in git history, screenshots and shared files. Instead:

1. Put them in a **`.env`** file in the project root:
   ```
   STARDOG_ENDPOINT=https://<your-endpoint>.stardog.cloud:5820
   STARDOG_USERNAME=<server user>
   STARDOG_PASSWORD=<password>
   STARDOG_DATABASE=<database name>
   ```
2. Make sure git ignores it: `.env` must be listed in `.gitignore`.
3. Load it with **python-dotenv**, which copies the values into `os.environ`.

> ⚠️ **The #1 beginner error:** `STARDOG_USERNAME` must be a user that exists **on the Stardog server** (created in Studio → Security → Users). Your cloud.stardog.com login email is a different account and gives `401 Unauthorized`.

`load_dotenv()` searches the current folder and its parents, so it finds the `.env` in the project root.

In [2]:
import os

from dotenv import load_dotenv

print('Loaded .env:', load_dotenv())

conn_details = {
    'endpoint': os.environ['STARDOG_ENDPOINT'],
    'username': os.environ['STARDOG_USERNAME'],
    'password': os.environ['STARDOG_PASSWORD'],
}
DB_NAME = os.environ['STARDOG_DATABASE']

# Print everything except the password
print({**conn_details, 'password': '****'}, '| database:', DB_NAME)

Loaded .env: True
{'endpoint': 'https://sd-fd55b7e2.stardog.cloud:5820', 'username': 'rakesh', 'password': '****'} | database: mysampledb


`conn_details` is a plain dictionary whose keys match the parameters of both `stardog.Admin` and `stardog.Connection`, so the same dictionary unpacks into either one with `**conn_details`.

The endpoint looks like this:

| Server | Endpoint |
|---|---|
| Stardog Cloud | `https://sd-xxxxxxxx.stardog.cloud:5820` |
| Local server | `http://localhost:5820` (pystardog's default) |

## 2.3 Two entry points: `Admin` and `Connection`

| | `stardog.Admin(**conn_details)` | `stardog.Connection(database, **conn_details)` |
|---|---|---|
| Scope | The whole server | One database |
| Typical tasks | Health, databases, options, users, roles, stored queries, virtual graphs | Load data, SPARQL queries and updates, transactions, reasoning, validation |
| Contacts the server | **Immediately** (checks `/admin/alive`), so bad credentials fail here | **Lazily**, on the first real request |
| Analogy | DBA console | Application session |

> 🔁 **Neo4j equivalent:** `GraphDatabase.driver(uri, auth=(user, pw))` plays the role of the connection details, and `driver.session(database='neo4j')` is like `stardog.Connection`. Neo4j's admin tasks run as Cypher on the `system` database; Stardog has a separate `Admin` API for them.

## 2.4 Server tasks with `Admin`

### Health and databases

In [3]:
with stardog.Admin(**conn_details) as admin:
    print('alive:      ', admin.alive())
    print('healthcheck:', admin.healthcheck())
    print('databases:  ', [db.name for db in admin.databases()])

alive:       True


healthcheck: True


databases:   ['catalog', 'querylog', 'mysampledb']


`with` makes sure the client's HTTP session is closed at the end, even if an error occurs. More on this in 2.6.

`catalog` and `querylog` are **system databases** that Stardog Cloud creates. Your data lives in your own database.

### Database options

Every database has **options** that switch features on or off. Later chapters use several of them:

In [4]:
import pandas as pd

OPTIONS = ['reasoning.type', 'reasoning.schemas', 'search.enabled', 'spatial.enabled',
           'edge.properties', 'security.named.graphs', 'query.all.graphs']

with stardog.Admin(**conn_details) as admin:
    db = admin.database(DB_NAME)
    options = db.get_options(*OPTIONS)
pd.DataFrame({'option': options.keys(), 'value': [str(v) for v in options.values()]})

,option,value
0,query.all.graphs,False
1,reasoning.type,SL
2,spatial.enabled,False
3,security.named.graphs,False
4,search.enabled,False
5,reasoning.schemas,"['Recipe_Basic=urn:Recipe_Basic:Recipe_Basic',..."
6,edge.properties,False


| Option | Meaning | Chapter |
|---|---|---|
| `reasoning.type` | Which reasoning profile is used (`SL` = Stardog's default) | 8 |
| `reasoning.schemas` | Named models (schemas) available for reasoning | 7, 8 |
| `search.enabled` | Full-text search | 12 |
| `spatial.enabled` | Geospatial queries | 12 |
| `edge.properties` | RDF-star edge properties | 1 |
| `security.named.graphs` | Per-graph permissions | 14 |
| `query.all.graphs` | Whether queries without `FROM` see all named graphs (off: only the default graph) | 3 |

### Namespaces

Each database stores **prefixes** that are available in every query without declaring them:

In [5]:
with stardog.Admin(**conn_details) as admin:
    namespaces = admin.database(DB_NAME).namespaces()
pd.DataFrame(namespaces)

,prefix,name
0,Recipe_Basic,urn:Recipe_Basic:
1,,http://api.stardog.com/
2,stardog,tag:stardog:api:
3,owl,http://www.w3.org/2002/07/owl#
4,rdf,http://www.w3.org/1999/02/22-rdf-syntax-ns#
5,xsd,http://www.w3.org/2001/XMLSchema#
6,rdfs,http://www.w3.org/2000/01/rdf-schema#
7,so,https://schema.org/


### Users, roles and your own permissions

Stardog has **users**, **roles** (groups of permissions), and **permissions** of the form *action on resource* (for example `READ` on database `mydb`). Chapter 14 covers security in depth; here we just look.

In [6]:
with stardog.Admin(**conn_details) as admin:
    print('users:', [u.name for u in admin.users()])
    print('roles:', [r.name for r in admin.roles()])
    me = admin.user(conn_details['username'])
    print(f"\n'{me.name}' has roles {[r.name for r in me.roles()]}; superuser: {me.is_superuser()}")
    my_permissions = pd.DataFrame(me.effective_permissions())
print(f'{len(my_permissions)} permissions in total. The ones on all resources (*):')
my_permissions[my_permissions.resource_type == '*'].reset_index(drop=True)

users: ['rakesh.panigrahy@chaatashaali.com', 'rakesh', 'admin']


roles: ['cloud', 'reader', 'executor']



'rakesh' has roles ['cloud']; superuser: False


93 permissions in total. The ones on all resources (*):


,action,resource_type,resource
0,WRITE,*,[*]
1,REVOKE,*,[*]
2,GRANT,*,[*]
3,CREATE,*,[*]
4,EXECUTE,*,[*]
5,DELETE,*,[*]
6,READ,*,[*]


`resource_type = *` and `resource = ['*']` mean "on everything". The `cloud` role gives your user broad rights: read, write, create and delete on all resources. The remaining rows are narrower permissions on specific databases and objects.

## 2.5 Creating and dropping databases, safely

The API is simple:

```python
db = admin.new_database('name')   # create
db.drop()                         # delete, with all its data
```

But Stardog Cloud plans can **limit how many databases** you have. When the limit is reached, `new_database` fails with `403`, even though you have `CREATE` permission, because the limit is a plan quota, not a permission. (Earlier in this project exactly that happened, until an unused database was deleted.)

So we write a **guarded** helper: try to create a temporary database, report what happened, and always clean up.

In [7]:
from stardog.exceptions import StardogException

def try_temp_database(name='tutorial_ch02_tmp'):
    with stardog.Admin(**conn_details) as admin:
        if name in [db.name for db in admin.databases()]:
            admin.database(name).drop()               # leftover from an earlier run
        try:
            db = admin.new_database(name)
        except StardogException as e:
            print(f'Could not create {name!r}: HTTP {e.http_code}')
            print('->', e)
            return False
        try:
            print('Created:', [d.name for d in admin.databases()])
        finally:
            db.drop()                                 # always remove it again
        print('Dropped:', [d.name for d in admin.databases()])
        return True

created = try_temp_database()
print()
print('This endpoint can create another database right now:', created)

Created: ['catalog', 'querylog', 'mysampledb', 'tutorial_ch02_tmp']


Dropped: ['catalog', 'querylog', 'mysampledb']

This endpoint can create another database right now: True


If you see *Created* and *Dropped*, your plan has room for another database. If you see a `403`, a limit was reached.

**Why does the tutorial use named graphs instead of a database per chapter?** Each chapter writes to its own graphs (`urn:tutorial:ch02:...`) inside `STARDOG_DATABASE` and clears them at the end. This works on every plan whatever its limits, needs no `CREATE` permission, is instant, and lets later chapters query several graphs together. Chapter 3 covers named graphs in detail.

> ⚠️ `db.drop()` deletes a database and **all** its data immediately. There's no recycle bin. Never call it on your main database.

## 2.6 Connections and their lifecycle

### Connections are lazy

Creating a `Connection` doesn't contact the server. Even a database name that doesn't exist is "accepted" until you use it:

In [8]:
typo = stardog.Connection('mysampledbb', **conn_details)
print('Created a connection object, no error yet')
try:
    typo.size()
except StardogException as e:
    print(f'First real request fails: HTTP {e.http_code} {e.stardog_code}: {e}')
finally:
    typo.close()

Created a connection object, no error yet


First real request fails: HTTP 404 0D0DU2: [404] 0D0DU2: Database 'mysampledbb' does not exist.


That's why the setup checker in 2.9 makes a real request (`size()`) to prove a connection works.

### Opening and closing

Two styles, same result:

In [9]:
# 1. Context manager: closed automatically, even on errors. Best for scripts and functions.
with stardog.Connection(DB_NAME, **conn_details) as conn:
    print('approx. triples:', conn.size())
    print('exact triples:  ', conn.size(exact=True))

# 2. Explicit: handy in notebooks where one connection is used across many cells
conn = stardog.Connection(DB_NAME, **conn_details)
print('still usable:   ', conn.size())
conn.close()

approx. triples: 2510


exact triples:   1887


still usable:    2510


`size()` is a fast **estimate**; `size(exact=True)` counts precisely, which is slower on big databases.

**Rules of thumb**

- **Scripts and APIs:** open a connection per unit of work with `with`.
- **Notebooks:** open one connection near the top, and close it in the clean-up cell.
- **Don't share one connection between threads**: it wraps one HTTP session and, once you call `begin()`, one transaction. Give each thread its own.

> 🔁 **Neo4j equivalent:** `with driver.session() as session:` is the same pattern as `with stardog.Connection(...) as conn:`.

## 2.7 When things go wrong

Every server-side error is raised as a **`StardogException`** with two useful attributes:

- `http_code`: the HTTP status (401, 403, 404, 400, …)
- `stardog_code`: Stardog's own error code, handy for searching the docs

Network problems (wrong host, no internet) come from the `requests` library instead, as `requests.exceptions.ConnectionError`.

Let's trigger each one on purpose:

In [10]:
import requests

def show_error(label, fn):
    try:
        fn()
        print(f'{label:18} no error')
    except StardogException as e:
        print(f'{label:18} StardogException  http={e.http_code}  code={e.stardog_code}  {str(e)[:70]}')
    except requests.exceptions.ConnectionError as e:
        print(f'{label:18} ConnectionError    {str(e)[:70]}')

show_error('wrong password', lambda: stardog.Admin(**{**conn_details, 'password': 'wrong'}))
show_error('unknown database', lambda: stardog.Connection('no_such_db', **conn_details).size())
show_error('bad SPARQL', lambda: stardog.Connection(DB_NAME, **conn_details).select('SELEC ?x'))
show_error('unknown host', lambda: stardog.Admin(**{**conn_details, 'endpoint': 'https://no-such-host.invalid:5820'}))

wrong password     StardogException  http=401  code=None  [401] : 


unknown database   StardogException  http=404  code=0D0DU2  [404] 0D0DU2: Database 'no_such_db' does not exist.


bad SPARQL         StardogException  http=400  code=QE0PE2  [400] QE0PE2: com.complexible.stardog.plan.eval.ExecutionException: En
unknown host       ConnectionError    HTTPSConnectionPool(host='no-such-host.invalid', port=5820): Max retri


| Error | Usual cause | Fix |
|---|---|---|
| **401** Unauthorized | Wrong user/password, or using the cloud.stardog.com login | Use a **server** user from Studio → Security → Users |
| **403** Forbidden | Missing permission, or a plan limit (e.g. database count) | Check permissions (2.4) or the plan |
| **404** Not found | Database name typo, or the database was dropped | Check `STARDOG_DATABASE` against `admin.databases()` |
| **400** Bad request | SPARQL syntax error | Read the message: it shows the position of the problem |
| `ConnectionError` | Wrong endpoint, missing port, no network | `https://…stardog.cloud:5820` |

## 2.8 A reusable client module: `kg.client`

Copying `load_dotenv` and `conn_details` into every notebook gets old fast. The tutorial keeps them in **`src/kg/client.py`**, which every later chapter imports. It provides:

| Function | Returns |
|---|---|
| `load_settings()` | A `Settings` object from `.env`, with a clear error if something is missing. Its `repr` hides the password. |
| `admin()` | A `stardog.Admin` |
| `connect(database=None)` | A `stardog.Connection` to `STARDOG_DATABASE` (or another database) |
| `explain_error(e)` | Advice for the errors in 2.7 |
| `check_setup()` | Runs all the checks a chapter needs (2.9) |

Notebooks live in `notebooks/`, so we add `../src` to Python's import path first:

In [11]:
import sys
from pathlib import Path

SRC = str(Path('..', 'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from kg import client

print(Path(client.__file__).read_text())

"""Stardog connection helpers used throughout the tutorial.

    from kg import client

    settings = client.load_settings()
    with client.admin() as admin: ...
    with client.connect() as conn: ...
"""
import os
from dataclasses import dataclass
from pathlib import Path

import requests
import stardog
from dotenv import load_dotenv

REQUIRED = ('STARDOG_ENDPOINT', 'STARDOG_USERNAME', 'STARDOG_PASSWORD', 'STARDOG_DATABASE')


@dataclass(frozen=True)
class Settings:
  endpoint: str
  username: str
  password: str
  database: str

  def conn_details(self) -> dict:
    """Keyword arguments accepted by stardog.Admin and stardog.Connection."""
    return {'endpoint': self.endpoint, 'username': self.username, 'password': self.password}

  def __repr__(self):
    # Never show the password, e.g. when a notebook displays the object
    return (f"Settings(endpoint='{self.endpoint}', username='{self.username}', "
            f"password='****', database='{self.database}')")


def find_env_file

Using it:

In [12]:
settings = client.load_settings()
print(settings)                         # password hidden

with client.admin() as admin:
    print('alive:', admin.alive())

with client.connect() as conn:
    print('triples in', settings.database, ':', conn.size())

try:
    stardog.Admin(**{**settings.conn_details(), 'password': 'wrong'})
except StardogException as e:
    print('\n', client.explain_error(e))

Settings(endpoint='https://sd-fd55b7e2.stardog.cloud:5820', username='rakesh', password='****', database='mysampledb')


alive: True


triples in mysampledb : 2510



 Login failed: STARDOG_USERNAME must be a user on the Stardog server, not your cloud.stardog.com login.


## 2.9 Hands-on: the setup checker

`client.check_setup()` runs every check a chapter depends on, in order, and stops at the first failure so the message points at the real cause:

1. Settings present in `.env`
2. Server reachable and login accepted
3. The database exists
4. Read access
5. Write access: writes a triple into a temporary named graph and removes it

In [13]:
pd.DataFrame(client.check_setup(), columns=['check', 'passed', 'detail'])

,check,passed,detail
0,Settings in .env,True,Settings(endpoint='https://sd-fd55b7e2.stardog...
1,Server reachable and login OK,True,alive = True
2,Database exists,True,'mysampledb' found
3,Read access,True,2510 triples (approx.)
4,Write access,True,wrote and removed <urn:tutorial:setup-check>


The same checker is a command-line script, which is handy after changing `.env` or on a new machine:

```
.venv\Scripts\python stardog_tutorial\00_setup\check_setup.py
```

In [14]:
import subprocess

result = subprocess.run([sys.executable, str(Path('..', '00_setup', 'check_setup.py'))],
                        capture_output=True, text=True)
print(result.stdout)
print('exit code:', result.returncode, '(0 = everything passed)')

PASS  Settings in .env                 Settings(endpoint='https://sd-fd55b7e2.stardog.cloud:5820', username='rakesh', password='****', database='mysampledb')
PASS  Server reachable and login OK    alive = True
PASS  Database exists                  'mysampledb' found
PASS  Read access                      2510 triples (approx.)
PASS  Write access                     wrote and removed <urn:tutorial:setup-check>

exit code: 0 (0 = everything passed)


### Put it to work

Load the Chapter 1 dataset through the client module and count the people, then clean up. This is the pattern every later chapter follows.

In [15]:
GRAPH = 'urn:tutorial:ch02:company'
DATA = Path('..', 'data', 'company', 'company.ttl')

with client.connect() as conn:
    conn.begin()
    conn.clear(graph_uri=GRAPH)                       # never clear() without graph_uri
    conn.add(stardog.content.File(str(DATA)), graph_uri=GRAPH)
    conn.commit()

    res = conn.select(f'''
        PREFIX co: <http://example.org/company#>
        SELECT (COUNT(?p) AS ?people) FROM <{GRAPH}> {{ ?p a co:Person }}''')
    print('People:', res['results']['bindings'][0]['people']['value'])

    conn.begin()
    conn.clear(graph_uri=GRAPH)
    conn.commit()
    print('Removed', GRAPH)

People: 5


Removed urn:tutorial:ch02:company


## 2.10 Exercises

**Exercise 1.** Print every database on the server with its approximate number of triples.

<details><summary>Solution</summary>

```python
with client.admin() as admin:
    names = [db.name for db in admin.databases()]
for name in names:
    with client.connect(database=name) as conn:
        print(f'{name:12} {conn.size():>8}')
```
</details>

**Exercise 2.** Write `graph_exists(conn, graph_uri) -> bool` that tells whether a named graph contains any triples. *Hint:* an `ASK` query with a `GRAPH` block.

<details><summary>Solution</summary>

```python
def graph_exists(conn, graph_uri):
    return conn.ask(f'ASK {{ GRAPH <{graph_uri}> {{ ?s ?p ?o }} }}')

with client.connect() as conn:
    print(graph_exists(conn, 'urn:tutorial:ch02:company'))   # False after clean-up
```
</details>

**Exercise 3.** Why did `stardog.Connection('mysampledbb', ...)` not fail, while `stardog.Admin(...)` with a wrong password failed immediately?

<details><summary>Solution</summary>

`Admin` calls `/admin/alive` in its constructor to check the server, so bad credentials fail at once. `Connection` only stores the details; the first request that needs the database (`size()`, `select()`, …) is what fails, with `404`.
</details>

**Exercise 4.** Make `check_setup()` fail on purpose by passing settings with a wrong database name, and read the advice it gives.

<details><summary>Solution</summary>

```python
from dataclasses import replace
bad = replace(client.load_settings(), database='nope')
pd.DataFrame(client.check_setup(bad), columns=['check', 'passed', 'detail'])
# 'Database exists' fails with: Not found: check the database name (STARDOG_DATABASE).
```
</details>

In [16]:
# your answers here

## Summary

- **pystardog** has two entry points: `Admin` for the server and `Connection` for one database. Both take the same `endpoint` / `username` / `password`.
- Credentials live in **`.env`**, loaded with python-dotenv, and never in code. The username must be a **server** user.
- `Admin` checks the server immediately; `Connection` is **lazy**, so make a real request to test it.
- Use **`with`** to close clients reliably.
- Errors arrive as `StardogException` with **`http_code`** (401/403/404/400) and **`stardog_code`**; network problems are `requests` `ConnectionError`s.
- Creating databases can hit a **plan quota** (`403`), so the tutorial isolates work with **named graphs** instead, which works on any plan.
- **`kg.client`** and **`check_setup.py`** are reused by every later chapter.

### Cheat sheet

```python
from kg import client

with client.admin() as admin:            # server
    admin.databases(); admin.users(); admin.database(name).get_options('reasoning.type')

with client.connect() as conn:           # database
    conn.size(); conn.select('...'); conn.begin(); conn.add(...); conn.commit()
```

**Next: Chapter 3, RDF data and loading.** Serialization formats, literals and datatypes, blank nodes, named graphs, and loading the Recipes dataset.